In [1]:
import argparse
import re
import glob
import time

import pandas as pd
from tbparse import SummaryReader
from torch.utils.tensorboard import SummaryWriter

# Important - should include at least every seed
log_dir_reg = '/home/rp218/projects/rm-marl/logs/test_parse/*'
# log_dir_reg = '/gpfs/home/rp218/rm-marl/saved_logs/*'
# Match any file name that start with some text and end with _{seed}_{sensor_confidence}
file_pattern = r"(.*)_(?P<seed>\d+)_(?P<sensor_confidence>[\d.]+)"



In [2]:
def to_dataframe(log_dirs, extraction_pattern):
    dfs = []
    for logdir in log_dirs:
        reader = SummaryReader(logdir)
        df = reader.scalars
        match = re.match(extraction_pattern, logdir)
        df['logdir'] = logdir
        df['seed'] = int(match.group("seed"))
        df['sensor_confidence'] = float(match.group("sensor_confidence"))
        dfs.append(df)
    return pd.concat(dfs)



In [15]:
def aggregate_by_tag(df):
    return df.groupby(['tag', 'step', 'sensor_confidence']).agg({'value': ['mean', 'std']}).reset_index()



In [4]:
def aggregate_by_env(df):
    df['tag_without_env'] = df['tag'].str.rsplit('/', n=1).str[0]
    return df.groupby(['tag_without_env', 'step', 'sensor_confidence']).agg({'value': ['mean', 'std']}).reset_index()



In [1]:
def write_tagged_results(grouped_df, tag_column_name, output_dir):
    writer = SummaryWriter(output_dir)
    # tags - eval/success_rate/* eval/failure_rate/*
    for tag in grouped_df[tag_column_name].unique():
        val = grouped_df.loc[grouped_df[tag_column_name] == tag]
        for index, row in val.iterrows():
            curr_step = row["step"].iloc[0]
            writer.add_scalar(f"{tag}/mean", row['value']['mean'], curr_step)

    writer.close()

In [6]:

log_dirs = glob.glob(log_dir_reg)
all_df = to_dataframe(log_dirs, file_pattern)
all_df

,step,tag,value,logdir,seed,sensor_confidence
0,1,eval/failure_rate/E0,0.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,223,1.0
1,2,eval/failure_rate/E0,0.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,223,1.0
2,3,eval/failure_rate/E0,0.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,223,1.0
3,4,eval/failure_rate/E0,0.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,223,1.0
4,5,eval/failure_rate/E0,0.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,223,1.0
...,...,...,...,...,...,...
560395,7996,training/tot_steps/E9,99841.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,433,1.0
560396,7997,training/tot_steps/E9,99850.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,433,1.0
560397,7998,training/tot_steps/E9,99861.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,433,1.0
560398,7999,training/tot_steps/E9,99874.0,/home/rp218/projects/rm-marl/logs/test_parse/c...,433,1.0


In [7]:
grouped_result = aggregate_by_tag(all_df)
grouped_result

tag  step sensor_confidence     value              
                                                           mean           std
0        eval/failure_rate/E0     1               1.0      0.00      0.000000
1        eval/failure_rate/E0     2               1.0      0.00      0.000000
2        eval/failure_rate/E0     3               1.0      0.00      0.000000
3        eval/failure_rate/E0     4               1.0      0.00      0.000000
4        eval/failure_rate/E0     5               1.0      0.00      0.000000
...                       ...   ...               ...       ...           ...
796965  training/tot_steps/E9  7996               1.0  84974.50  42643.543962
796966  training/tot_steps/E9  7997               1.0  84984.25  42648.387061
796967  training/tot_steps/E9  7998               1.0  84994.50  42652.698082
796968  training/tot_steps/E9  7999               1.0  85005.50  42657.274686
796969  training/tot_steps/E9  8000               1.0  85017.25  42662.572241

[796970 rows x 5 columns]

In [13]:
write_tagged_results(grouped_result, tag_column_name="tag", output_dir="summary/")

In [9]:
env_grouped_results = aggregate_by_env(all_df)

In [14]:
write_tagged_results(env_grouped_results, tag_column_name="tag_without_env", output_dir="eval/summary")

In [11]:
env_grouped_results

tag_without_env  step sensor_confidence       value               
                                                         mean            std
0       eval/failure_rate     1               1.0       0.075       0.266747
1       eval/failure_rate     2               1.0       0.025       0.158114
2       eval/failure_rate     3               1.0       0.000       0.000000
3       eval/failure_rate     4               1.0       0.025       0.158114
4       eval/failure_rate     5               1.0       0.000       0.000000
...                   ...   ...               ...         ...            ...
79692  training/tot_steps  7996               1.0  300425.300  234511.080736
79693  training/tot_steps  7997               1.0  300453.925  234527.673026
79694  training/tot_steps  7998               1.0  300477.275  234536.786835
79695  training/tot_steps  7999               1.0  300507.850  234551.801904
79696  training/tot_steps  8000               1.0  300533.750  234567.782880

[79697 rows x 5 columns]